# Continue projected finite-difference GD: up to 10 total iterations

**Preserved pilot → resume its last accepted mask → continue GD → validate → compare.**

This is a separate continuation of [the two-step GD notebook](dnl_gradient_descent_spectral_mask.ipynb).
The original notebook and raw run are preserved. The saved pilot reached accepted step 2;
this branch attempts steps **3–10**, using the same physics, objective, 0.01 factor grid
and acceptance rules. It may stop earlier if no feasible projected improvement is found.

Select **GNLSE (Python)**. Details are in
[the continuation helper](../../../notebook_helpers/llm_bo/spectral_mask_gd_continue.py).

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebook_helpers" / "run_library.py").is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebook_helpers"))
from llm_bo.spectral_mask_gd_continue import continue_experiment
from llm_bo.spectral_mask_gd_continue_report import show_continuation_summary
from llm_bo.spectral_mask_run_costs import show_run_costs
from llm_bo.spectral_mask_gd_report import (
    show_descent_route, show_results, show_comparison, show_pulse_comparison,
)
get_ipython().run_line_magic("matplotlib", "inline")

## 1. Same physics, mask and starting point

The unmasked input is a **100 kW, 200 fs Gaussian**, with equal amplitudes and zero
relative phase in LP01, LP11-cos and LP11-sin. The original silica step-index fiber
is retained: 2.86 m length, 52.5 μm core radius, NA 0.1, and 1379.2 nm carrier.
The existing solver supplies the same dispersion, nonlinear response and CP overlap tensor.

The **100 equal-width angular-frequency bands** span the centred interval containing
99.9% of the original pulse's spectral energy. Their physical edges stay fixed.
One field-amplitude mask is shared by all three modes:

\[
\widetilde A_{\mathrm{shaped}}(\omega)=s_j\widetilde A(\omega),\qquad
s_j\in\{0,0.01,\ldots,1\}.
\]

Frequencies outside the controlled bandwidth are unchanged. There is **no
renormalization**: a factor of 0.5 retains 0.25 of that band's power. The transform
convention remains `fftshift(ifft(A))`, with inverse `fft(ifftshift(spectrum))`.

Minimize the coherent **all-mode D_NL** between matched linear (`n2=0`) and nonlinear
outputs, including all three modes and without temporal alignment. The output-photon
floor is **50% of the unmasked 1 mm baseline**. Each objective evaluation is a matched
pair of propagations. The baseline, seed and restoration measurements are inherited from the preserved pilot.

The search uses **Nt=16384, T=64.87042 ps, dz=5 mm**, with four samples per band.
The original seed is 20261113, identical to LLMBO. `LOAD_RUN` opens this saved continuation
without new simulations; set it to `None` to create another bounded branch from `PARENT_RUN`.

In [ ]:
PARENT_RUN = None
MAX_ITERATIONS = 10         # Total: the two saved steps plus up to eight new steps.
MAX_NEW_EVALUATIONS = 1700  # New objective pairs; inherited results and cache hits are free.
MAX_MINUTES = 180           # Additional calendar time, including validation.
VALIDATION_MINUTES = 6      # Reserved inside MAX_MINUTES.
LOAD_RUN = None
LLM_REFERENCE = None

## Continue from a two-step pilot

Set `PARENT_RUN` to the directory printed by the pilot, or leave it empty to run a fresh pilot first. Set `LOAD_RUN` to reopen a finished continuation.

In [ ]:
# File the saved campaign under its launch controls and search method.
from notebook_helpers.run_library import file_saved_run

if PARENT_RUN is None and LOAD_RUN is None:
    from llm_bo.spectral_mask_gd import run_experiment
    PARENT_RUN = run_experiment().directory

run = continue_experiment(
    parent_directory=PARENT_RUN, max_iterations=MAX_ITERATIONS,
    max_new_evaluations=MAX_NEW_EVALUATIONS, max_minutes=MAX_MINUTES,
    validation_minutes=VALIDATION_MINUTES, load_directory=LOAD_RUN,
)
file_saved_run(run.directory, "Spectral amplitude mask", "Gradient descent")

## 3. Descent route, selected mask and numerical checks

First show the saved stopping reason and the **accepted GD route**. Step 0 is the
first feasible mask after restoration. Rejected proposals are measured tests; they
do not advance the accepted iterate. The history plot connects only accepted GD iterates;
initialization, restoration and rejected proposals appear as separate markers.
Backtracking starts each proposal from the current accepted mask, not the previous
rejected proposal.

The remaining plots retain the input-energy and brightness histories, 100 factors,
input/output spectra, pulse centre and full-window temporal tails. All gradient
probes remain in the full saved evaluation ledger.

Recheck the unmasked baseline and selected mask at **dz=1 mm** on the search grid,
then at **Nt=65536, T=129.74084 ps**, which doubles the window and halves dt.
Use the same physical band edges and factors. The absolute D_NL tolerance is
**0.0002** for propagation-step and time/frequency-grid comparisons. The refined
brightness check uses the unmasked 1 mm reference on the same refined grid, as in LLMBO.

Failed or missing checks leave the result **preliminary**. Raw scores remain visible;
attenuation and pulse broadening are allowed controls and do not by themselves
establish improved performance for an application.

In [ ]:
summary = show_continuation_summary(run)
show_descent_route(run)
show_results(run)

## Compare with your spectral-mask LLM run (optional)

Set `LLM_REFERENCE` above to the run directory printed by the spectral-mask LLM notebook.

In [ ]:
costs = show_run_costs(run)
if LLM_REFERENCE is not None:
    comparison = show_comparison(run, LLM_REFERENCE)

### LLAMBO beside GD: input, linear output and nonlinear output

**LLAMBO on the left; GD on the right.** Each panel shows the selected **masked input**,
its **matched linear output** and its **nonlinear output**, using the saved 1 mm
propagation checks on the same grid. Compare the pulse centre, wavelength spectrum
and full time window on shared absolute scales, summing all three modes without
renormalization or time alignment. These are the selected shaped pulses; the common
100 kW Gaussian is the launch **before** shaping.

In [ ]:
if LLM_REFERENCE is not None:
    show_pulse_comparison(run, LLM_REFERENCE)